# Can a CNN still read obfuscated digits? (10-epoch check)

A small replication of McPherson, Shokri and Shmatikov (2016), *Defeating Image Obfuscation with Deep Learning*, on MNIST.

**How to run (Google Colab):** Runtime → Change runtime type → pick a GPU if one is offered → Save. Then Runtime → Run all. It does 90 epochs of training in total against 54 for the original run, so expect a little under twice its time. The results table appears at the end; download it from the Files panel (folder icon, left), or with the last cell.

**Disclosure:** this code was written by an AI coding assistant (Claude, by Anthropic). Say so in your paper.

**What this run checks:** the paper's 2-epoch model read 89.11% of digits blurred with σ = 4, while Privacy Blur (Mahloujifar et al., 2025) reported 97.58% at about the same blur. This run trains for 10 epochs instead of 2 on three conditions only: clean (needed first, for the clean-trained model), blur σ = 4, and pixelate 7×7 as a control. If more training closes the blur gap but barely moves 7×7, the paper's claim that pixelation is limited by the information it leaves holds up within one setup. Results go to `results_10epochs.csv`, so the original `results.csv` is not overwritten.

**After it finishes:** File → Download → Download .ipynb, so the saved notebook keeps its printed outputs (the public record for the GitHub repository).

In [1]:
import time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 10     # passes over the 60,000 training images
BATCH = 128     # images per training step
LR = 1e-3       # Adam learning rate
SEEDS = [0, 1, 2] if device == 'cuda' else [0]   # 3 repeats on a GPU, 1 on a CPU
print('Device:', device, '| seeds:', SEEDS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

Device: cuda | seeds: [0, 1, 2]


## 1. Data: MNIST (60,000 training and 10,000 test images of handwritten digits, 28×28 pixels)

In [2]:
train = torchvision.datasets.MNIST('data', train=True, download=True)
test = torchvision.datasets.MNIST('data', train=False, download=True)
Xtr = train.data.float().div(255).unsqueeze(1)   # 60000 x 1 x 28 x 28, pixel values 0..1
ytr = train.targets
Xte = test.data.float().div(255).unsqueeze(1)    # 10000 x 1 x 28 x 28
yte = test.targets
print('Train:', tuple(Xtr.shape), '| Test:', tuple(Xte.shape))

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.95MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.7MB/s]


Train: (60000, 1, 28, 28) | Test: (10000, 1, 28, 28)


## 2. The obfuscations

In [3]:
def pixelate(x, block):
    """Mosaic: average each block x block square, then scale it back up to 28 x 28."""
    return F.interpolate(F.avg_pool2d(x, block), scale_factor=block, mode='nearest')

def blur(x, sigma):
    """Gaussian blur with standard deviation sigma, in pixels."""
    radius = int(3 * sigma)
    t = torch.arange(-radius, radius + 1, dtype=torch.float32)
    g = torch.exp(-t ** 2 / (2 * sigma ** 2))
    g = g / g.sum()
    k = len(g)
    x = F.pad(x, (radius, radius, radius, radius), mode='reflect')
    x = F.conv2d(x, g.view(1, 1, 1, k))   # blur along each row
    x = F.conv2d(x, g.view(1, 1, k, 1))   # then along each column
    return x

def solid_mask(x):
    """Cover the whole digit with a solid black box: no pixel of it survives."""
    return torch.zeros_like(x)

CONDITIONS = {                     # 'clean' must stay first
    'clean': lambda x: x,
    'pixelate 7x7': lambda x: pixelate(x, 7),   # control: does pixelation move with more epochs?
    'blur sigma=4': lambda x: blur(x, 4),
}


## 3. The model: a small CNN (two convolution + pooling layers, then two fully connected layers)

In [4]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)    # 32 filters, 3x3
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)   # 64 filters, 3x3
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)                  # one output per digit

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)     # 28x28 -> 14x14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)     # 14x14 -> 7x7
        x = self.drop(F.relu(self.fc1(x.flatten(1))))
        return self.fc2(x)

def train_model(X, y, seed):
    set_seed(seed)
    model = SmallCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(X), device=X.device)
        for i in range(0, len(X), BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad()
            loss = F.cross_entropy(model(X[idx]), y[idx])
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    correct = sum((model(X[i:i + 1000]).argmax(1) == y[i:i + 1000]).sum().item()
                  for i in range(0, len(X), 1000))
    return 100.0 * correct / len(X)

## 4. The experiment

For each obfuscation, two models are tested on obfuscated test images:
- **Trained on clean images:** a model that has never seen obfuscation (a naive attacker).
- **Trained on obfuscated images:** a model trained on obfuscated images with their labels (the attacker from McPherson et al.).

In [5]:
ytr_d, yte_d = ytr.to(device), yte.to(device)
rows, clean_models = [], {}
t_start = time.time()
for name, f in CONDITIONS.items():
    t0 = time.time()
    Xtr_o, Xte_o = f(Xtr).to(device), f(Xte).to(device)
    naive, trained = [], []
    for s in SEEDS:
        model = train_model(Xtr_o, ytr_d, s)
        if name == 'clean':
            clean_models[s] = model
        trained.append(accuracy(model, Xte_o, yte_d))
        naive.append(accuracy(clean_models[s], Xte_o, yte_d))
    rows.append({'condition': name,
                 'trained_on_clean_%': np.mean(naive), 'trained_on_clean_sd': np.std(naive),
                 'trained_on_obfuscated_%': np.mean(trained), 'trained_on_obfuscated_sd': np.std(trained)})
    print(f'{name:15s} trained on clean: {np.mean(naive):5.1f}%   '
          f'trained on obfuscated: {np.mean(trained):5.1f}%   ({time.time() - t0:.0f} s)')
print(f'Total: {(time.time() - t_start) / 60:.1f} min')

clean           trained on clean:  99.2%   trained on obfuscated:  99.2%   (51 s)
pixelate 7x7    trained on clean:  33.1%   trained on obfuscated:  87.6%   (51 s)
blur sigma=4    trained on clean:  32.1%   trained on obfuscated:  94.9%   (60 s)
Total: 2.7 min


## 5. Results table (saved as results_10epochs.csv)

In [6]:
df = pd.DataFrame(rows).round(2)
df.to_csv('results_10epochs.csv', index=False)
df

,condition,trained_on_clean_%,trained_on_clean_sd,trained_on_obfuscated_%,trained_on_obfuscated_sd
0,clean,99.19,0.05,99.19,0.05
1,pixelate 7x7,33.05,1.69,87.64,0.57
2,blur sigma=4,32.05,4.62,94.89,0.13


## 6. Optional: download the results (Colab only)

In [7]:
try:
    from google.colab import files
    for name in ['results_10epochs.csv']:
        files.download(name)
except ImportError:
    print("Not running on Colab: the files are in this notebook's folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>